# Replication Notebook: Tables 2–4

**Paper:** Lai, T. B. (2025). Trading-Aware Agents in Sugarscape: A Deep Reinforcement Learning Approach.  
*Computational Economics* (Revision 1).

This notebook reproduces the three statistical tables added in Revision 1:

| Table | Content | Section |
|---|---|---|
| Table 2 | Kaplan–Meier survival analysis | §4.1 Carrying Capacity |
| Table 3 | Inequality measures (Gini, Theil's T, Top-10% share) | §4.4 Inequality |
| Table 4 | Terminal wealth distribution and KS test | §4.4 Inequality |

## Data
All inputs are pre-generated pickle files in `submission_data/`.  
Config mapping (confirmed from agent-count diagnostics):

| Config | Condition |
|---|---|
| config1 | Rule-based, Deterministic |
| config2 | DRL, Deterministic |
| config3 | Rule-based, Stochastic |
| config4 | DRL, Stochastic |

## Requirements
```
pip install lifelines scipy numpy pandas
```

In [1]:
import sys, warnings
sys.path.insert(0, '.')   # needed to unpickle simulation_manager dataclasses
import pickle
import numpy as np
import pandas as pd
from scipy import stats
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings('ignore')

DATA_DIR = 'submission_data'
T_MAX    = 150   # episode horizon
R        = 50    # replicates per condition

print('Libraries loaded.')
print(f'  lifelines {__import__("lifelines").__version__}')
print(f'  scipy     {__import__("scipy").__version__}')
print(f'  numpy     {np.__version__}')

Libraries loaded.
  lifelines 0.30.0
  scipy     1.15.3
  numpy     2.2.6


## 1  Load Data

In [2]:
def load_pkl(name):
    with open(f'{DATA_DIR}/{name}', 'rb') as f:
        return pickle.load(f)

# Per-step agent records: list[replicate] -> list[step] -> list[AgentData]
sa = {i: load_pkl(f'list_step_agents_config{i}.pkl') for i in range(1, 5)}

# Gini time series (official simulation output): list[replicate] -> list[step_gini]
iq = {i: load_pkl(f'list_inequality_config{i}.pkl')   for i in range(1, 5)}

print('Data loaded.')
for i in range(1, 5):
    n_rep   = len(sa[i])
    n_steps = len(sa[i][0])
    n_ag0   = len(sa[i][0][0])
    print(f'  config{i}: {n_rep} replicates × {n_steps} steps, {n_ag0} agents at step 0')

Data loaded.
  config1: 50 replicates × 150 steps, 498 agents at step 0
  config2: 50 replicates × 150 steps, 497 agents at step 0
  config3: 50 replicates × 150 steps, 496 agents at step 0
  config4: 50 replicates × 150 steps, 498 agents at step 0


## 2  Helper Functions

In [3]:
# ── Survival ──────────────────────────────────────────────────────────────────
def get_survival_data(cfg_data):
    """
    Reconstruct agent lifetimes from per-step records.
    Returns (durations, events) pooled across all replicates.
      duration = last simulation step at which the agent was observed
      event    = 1 (died before T_MAX) or 0 (right-censored, alive at T_MAX)
    """
    durations, events = [], []
    for rep in cfg_data:
        last_seen = {}
        for step_idx, agents in enumerate(rep):
            for ag in agents:
                last_seen[ag.agentId] = step_idx + 1   # 1-indexed step
        for aid, ls in last_seen.items():
            if ls < T_MAX:
                durations.append(ls);    events.append(1)
            else:
                durations.append(T_MAX); events.append(0)
    return np.array(durations), np.array(events)


def get_median_ci(kmf, label):
    """
    Extract 95 % CI for median survival from the Greenwood confidence band.
      Lower bound: first time the lower KM band drops below 0.5
      Upper bound: first time the upper KM band drops below 0.5
    """
    ci  = kmf.confidence_interval_
    lo  = ci[f'{label}_lower_0.95'].values
    hi  = ci[f'{label}_upper_0.95'].values
    tl  = ci.index.values
    lo_idx = np.where(lo < 0.5)[0]
    hi_idx = np.where(hi < 0.5)[0]
    ci_lo = float(tl[lo_idx[0]]) if len(lo_idx) else np.nan
    ci_hi = float(tl[hi_idx[0]]) if len(hi_idx) else np.inf
    return ci_lo, ci_hi


# ── Wealth ────────────────────────────────────────────────────────────────────
def terminal_wealth_per_rep(cfg_data):
    """Per-replicate list of agent terminal wealth (sugar + spice at T = 150)."""
    return [[ag.remainSugar + ag.remainSpice for ag in rep[-1]]
            for rep in cfg_data]


def pooled_terminal_wealth(cfg_data):
    """All agent terminal wealth pooled across 50 replicates."""
    return np.array([ag.remainSugar + ag.remainSpice
                     for rep in cfg_data for ag in rep[-1]])


# ── Inequality statistics ─────────────────────────────────────────────────────
def theil_t(wealth):
    """Theil's T entropy index on a wealth vector."""
    w = np.asarray(wealth, dtype=float)
    w = w[w > 0]
    if len(w) < 2: return np.nan
    r = w / w.mean()
    return float(np.mean(r * np.log(r)))


def top10_share(wealth):
    """Wealth share held by the top 10 % of agents (%)."""
    w     = np.sort(np.asarray(wealth, dtype=float))[::-1]
    top_n = max(1, int(np.ceil(0.1 * len(w))))
    return 100 * w[:top_n].sum() / w.sum() if w.sum() > 0 else np.nan


def cohens_d_paired(a, b):
    """Paired Cohen's d (SD of within-pair differences as denominator)."""
    diff = np.asarray(a) - np.asarray(b)
    return diff.mean() / diff.std(ddof=1)


print('Helper functions defined.')

Helper functions defined.


## 3  Table 2 — Kaplan–Meier Survival Analysis

Agent lifetimes are reconstructed by recording the last step at which each `agentId` 
appears in the per-step records.  Agents present at step 150 are right-censored.

In [4]:
CONFIGS = [
    (1, 'Rule-based', 'Deterministic'),
    (2, 'DRL',        'Deterministic'),
    (3, 'Rule-based', 'Stochastic'),
    (4, 'DRL',        'Stochastic'),
]

km_rows = []
km_store = {}   # keep KMF objects for log-rank test

for cfg, policy, landscape in CONFIGS:
    label = f'{policy}-{landscape[:3]}'
    dur, ev = get_survival_data(sa[cfg])
    kmf = KaplanMeierFitter()
    kmf.fit(dur, ev, label=label)

    med   = float(kmf.median_survival_time_)
    ci_lo, ci_hi = get_median_ci(kmf, label)
    sf    = float(kmf.survival_function_at_times([T_MAX]).values[0]) * 100
    ci_hi_str = f'{ci_hi:.0f}' if ci_hi != np.inf else 'n.r.'

    km_rows.append({
        'Policy'    : policy,
        'Landscape' : landscape,
        'N (pooled)': len(dur),
        'Deaths n (%)': f"{int(ev.sum())} ({100*ev.sum()/len(dur):.1f})",
        'Median [95 % CI]': f"{med:.0f} [{ci_lo:.0f}–{ci_hi_str}]",
        'S(150) %'  : round(sf, 1),
    })
    km_store[cfg] = dict(dur=dur, ev=ev, med=med)

# Log-rank tests
lr_det = logrank_test(km_store[1]['dur'], km_store[2]['dur'],
                      km_store[1]['ev'],  km_store[2]['ev'])
lr_sto = logrank_test(km_store[3]['dur'], km_store[4]['dur'],
                      km_store[3]['ev'],  km_store[4]['ev'])

df_km = pd.DataFrame(km_rows)
display(df_km)

print('\nLog-rank tests (DRL vs. Rule-based):')
for land, lr, rb, drl in [('Deterministic', lr_det, 1, 2), ('Stochastic', lr_sto, 3, 4)]:
    pct = 100 * (km_store[drl]['med'] - km_store[rb]['med']) / km_store[rb]['med']
    print(f'  {land}: χ² = {lr.test_statistic:.1f}, p = {lr.p_value:.2e}, '
          f'median change = +{pct:.0f}%')

,Policy,Landscape,N (pooled),Deaths n (%),Median [95 % CI],S(150) %
0,Rule-based,Deterministic,24783,20064 (81.0),11 [11–11],19.0
1,DRL,Deterministic,24762,18746 (75.7),16 [16–17],24.3
2,Rule-based,Stochastic,24810,19100 (77.0),18 [17–18],23.0
3,DRL,Stochastic,24894,17414 (70.0),26 [26–27],30.0



Log-rank tests (DRL vs. Rule-based):
  Deterministic: χ² = 727.6, p = 3.05e-160, median change = +45%
  Stochastic: χ² = 591.4, p = 1.22e-130, median change = +44%


## 4  Table 3 — Inequality Measures

- **Gini**: taken from the simulation output (`list_inequality_configN.pkl`), terminal step value.
- **Theil's T**: computed from individual agent wealth (sugar + spice) at T = 150.
- **Top-10% wealth share**: computed from the same individual wealth vector.

Statistics are per-replicate (n = 50), compared with a paired *t*-test.

In [5]:
# Per-replicate inequality statistics
gini_rep  = {i: np.array([rep[-1] for rep in iq[i]])  for i in range(1, 5)}

wealth_rep = {i: terminal_wealth_per_rep(sa[i]) for i in range(1, 5)}
theil_rep  = {i: np.array([theil_t(w)     for w in wealth_rep[i]]) for i in range(1, 5)}
top10_rep  = {i: np.array([top10_share(w) for w in wealth_rep[i]]) for i in range(1, 5)}

ineq_rows = []
for rb, drl, landscape in [(1, 2, 'Deterministic'), (3, 4, 'Stochastic')]:
    for arr_rb, arr_drl, measure in [
        (gini_rep[rb],  gini_rep[drl],  'Gini coefficient'),
        (theil_rep[rb], theil_rep[drl], "Theil's T index"),
        (top10_rep[rb], top10_rep[drl], 'Top-10% wealth share'),
    ]:
        t_stat, p_val = stats.ttest_rel(arr_drl, arr_rb)
        d = cohens_d_paired(arr_drl, arr_rb)
        ineq_rows.append({
            'Landscape' : landscape,
            'Measure'   : measure,
            'Rule-based (mean ± SD)': f'{arr_rb.mean():.3f} ± {arr_rb.std(ddof=1):.3f}',
            'DRL (mean ± SD)'       : f'{arr_drl.mean():.3f} ± {arr_drl.std(ddof=1):.3f}',
            't(49)'  : round(t_stat, 2),
            'p'      : '<0.001' if p_val < 0.001 else round(p_val, 3),
            'd'      : round(d, 2),
        })

df_ineq = pd.DataFrame(ineq_rows)
display(df_ineq.to_string(index=False))

"    Landscape              Measure Rule-based (mean ± SD) DRL (mean ± SD)  t(49)      p     d\nDeterministic     Gini coefficient          0.329 ± 0.031   0.327 ± 0.021  -0.38  0.702 -0.05\nDeterministic      Theil's T index          0.187 ± 0.039   0.185 ± 0.022  -0.31  0.759 -0.04\nDeterministic Top-10% wealth share         22.644 ± 2.232  21.760 ± 1.514  -2.51  0.015 -0.36\n   Stochastic     Gini coefficient          0.361 ± 0.022   0.364 ± 0.023   0.71  0.484  0.10\n   Stochastic      Theil's T index          0.185 ± 0.025   0.207 ± 0.027   4.45 <0.001  0.63\n   Stochastic Top-10% wealth share         22.799 ± 1.334  23.199 ± 1.524   1.36   0.18  0.19"

## 5  Table 4 — Terminal Wealth Distribution and KS Test

Agent wealth (sugar + spice) at episode termination is pooled across all 50 replicates.
The two-sided Kolmogorov–Smirnov test compares DRL vs. rule-based distributions.

In [6]:
pooled = {i: pooled_terminal_wealth(sa[i]) for i in range(1, 5)}

ks_rows = []
for rb, drl, landscape in [(1, 2, 'Deterministic'), (3, 4, 'Stochastic')]:
    w_rb  = pooled[rb]
    w_drl = pooled[drl]
    ks    = stats.ks_2samp(w_rb, w_drl)

    pcts_rb  = np.percentile(w_rb,  [50, 75, 90, 95, 99])
    pcts_drl = np.percentile(w_drl, [50, 75, 90, 95, 99])

    for policy, w, pcts in [('Rule-based', w_rb, pcts_rb), ('DRL', w_drl, pcts_drl)]:
        ks_rows.append({
            'Landscape' : landscape,
            'Policy'    : policy,
            'n'         : len(w),
            'Mean ± SD' : f'{w.mean():.1f} ± {w.std(ddof=1):.1f}',
            'P50'       : round(pcts[0], 1),
            'P75'       : round(pcts[1], 1),
            'P90'       : round(pcts[2], 1),
            'P95'       : round(pcts[3], 1),
            'P99'       : round(pcts[4], 1),
            'KS D'      : round(ks.statistic, 4),
            'KS p'      : '<0.001' if ks.pvalue < 0.001 else round(ks.pvalue, 4),
        })

df_ks = pd.DataFrame(ks_rows)
display(df_ks.to_string(index=False))

'    Landscape     Policy    n     Mean ± SD   P50   P75   P90   P95   P99   KS D   KS p\nDeterministic Rule-based 4719 256.1 ± 158.2 237.0 352.0 461.0 510.1 685.8 0.0777 <0.001\nDeterministic        DRL 6016 227.0 ± 135.7 210.5 323.0 414.0 462.0 561.7 0.0777 <0.001\n   Stochastic Rule-based 5710 281.5 ± 170.3 264.0 383.0 504.0 589.6 756.0 0.0429 <0.001\n   Stochastic        DRL 7480 269.0 ± 172.9 251.0 368.0 494.0 565.0 787.6 0.0429 <0.001'

## 6  Cross-check: Numbers Against Manuscript Tables

The cells below reproduce the exact values reported in the paper for quick verification.

In [7]:
print('=== Table 2 verification ===')
print(f'  RB-Det  median: {km_store[1]["med"]:.0f}   (paper: 11)')
print(f'  DRL-Det median: {km_store[2]["med"]:.0f}   (paper: 16)')
print(f'  RB-Sto  median: {km_store[3]["med"]:.0f}   (paper: 18)')
print(f'  DRL-Sto median: {km_store[4]["med"]:.0f}   (paper: 26)')
print(f'  Log-rank Det χ²: {lr_det.test_statistic:.1f}  (paper: 727.6)')
print(f'  Log-rank Sto χ²: {lr_sto.test_statistic:.1f}  (paper: 591.4)')

print('\n=== Table 3 verification ===')
# Theil T stochastic (key finding)
tT, pT = stats.ttest_rel(theil_rep[4], theil_rep[3])
print(f'  Theil T Sto DRL: {theil_rep[4].mean():.3f} ± {theil_rep[4].std(ddof=1):.3f}  (paper: 0.207 ± 0.027)')
print(f'  Theil T Sto RB:  {theil_rep[3].mean():.3f} ± {theil_rep[3].std(ddof=1):.3f}  (paper: 0.185 ± 0.025)')
print(f'  t(49) = {tT:.2f}, p = {pT:.4f}               (paper: t = 4.45, p < 0.001)')

print('\n=== Table 4 verification ===')
ks_det = stats.ks_2samp(pooled[1], pooled[2])
ks_sto = stats.ks_2samp(pooled[3], pooled[4])
print(f'  KS Det D = {ks_det.statistic:.4f}  (paper: 0.0777)')
print(f'  KS Sto D = {ks_sto.statistic:.4f}  (paper: 0.0429)')
print(f'  Det P99 RB = {np.percentile(pooled[1], 99):.1f}  (paper: 685.8)')
print(f'  Det P99 DRL= {np.percentile(pooled[2], 99):.1f}  (paper: 561.7)')

=== Table 2 verification ===
  RB-Det  median: 11   (paper: 11)
  DRL-Det median: 16   (paper: 16)
  RB-Sto  median: 18   (paper: 18)
  DRL-Sto median: 26   (paper: 26)
  Log-rank Det χ²: 727.6  (paper: 727.6)
  Log-rank Sto χ²: 591.4  (paper: 591.4)

=== Table 3 verification ===
  Theil T Sto DRL: 0.207 ± 0.027  (paper: 0.207 ± 0.027)
  Theil T Sto RB:  0.185 ± 0.025  (paper: 0.185 ± 0.025)
  t(49) = 4.45, p = 0.0000               (paper: t = 4.45, p < 0.001)

=== Table 4 verification ===
  KS Det D = 0.0777  (paper: 0.0777)
  KS Sto D = 0.0429  (paper: 0.0429)
  Det P99 RB = 685.8  (paper: 685.8)
  Det P99 DRL= 561.7  (paper: 561.7)
